# Projekt 5 — Uczenie przez wzmacnianie w przestrzeniach ciągłych

**Autorzy:** Konrad Małek, Bartłomiej Mazgaj

Niniejsze sprawozdanie realizuje **wszystkie zadania z poziomu 8 punktów** projektu 5:

1. Wybór środowiska Gymnasium z ciągłymi stanami i akcjami oraz algorytmu z `stable-baselines3`.
2. Krzywe uczenia dla **trzech zestawów hiperparametrów**, średnia ± odchylenie standardowe po **10 uruchomieniach** na każdy zestaw, **200 000 kroków czasowych** na uruchomienie.
3. Schematy **dwóch różnych architektur** sieci agenta wraz z opisem wejścia, wyjścia, warstw i funkcji aktywacji.
4. Zapis najlepszego modelu, **deterministyczna ewaluacja** (wyłączona eksploracja) i porównanie nagród z krzywą uczenia.

## 1. Środowisko – `WarehouseRobotEnv`

Środowisko `WarehouseRobotEnv` symuluje robota magazynowego w przestrzeni 2D o wymiarach 10×10 jednostek. Agent steruje robotem zadając w każdym kroku wektor przyspieszenia $a \in [-1, 1]^2$ (ciągła przestrzeń akcji `Box((2,))`), a w odpowiedzi otrzymuje 17-wymiarowy wektor obserwacji `Box((17,))` zawierający znormalizowaną pozycję i prędkość robota, flagę niesienia paczki, wektor i odległość do bieżącego celu, 8 odczytów z radialnych czujników odległości (co 45°) oraz odsetek pozostałych paczek. Robot musi zebrać 3 paczki (zbliżając się i zwalniając poniżej $v_\text{pickup}$) i dostarczyć je do strefy bazy, omijając dwie prostokątne półki. Funkcja nagrody łączy karę za krok (-0.1), karę za kolizję (-5), bonusy za podniesienie (+20) i dostarczenie (+100) paczki, bonus za ukończenie (+200) oraz **reward shaping** w postaci $\beta\cdot(d_\text{prev} - d_\text{curr})$ względem aktualnego celu. Limit epizodu wynosi 500 kroków. Na potrzeby tego projektu w środowisku uruchomiona jest **pula 5 deterministycznych layoutów** (parametr `layout_pool=LAYOUT_POOL_5`) — przy każdym `reset()` losowany jest jeden z pięciu zestawów pozycji paczek, co daje umiarkowaną generalizację bez pełnej losowości.

## 2. Algorytm – PPO (`stable-baselines3`)

**Proximal Policy Optimization (PPO)** to algorytm policy-gradient z klasy actor-critic, który aktualizuje politykę przy pomocy próbek z bufora rollouts, ale ogranicza wielkość kroku polityki przez tzw. *clipped surrogate objective* $L^{\text{CLIP}} = \mathbb{E}\!\left[\min(r_t(\theta) \hat{A}_t,\, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t)\right]$, gdzie $r_t$ to stosunek prawdopodobieństw nowej i starej polityki, a $\hat{A}_t$ to oszacowanie *advantage* (tu liczone metodą GAE). Aktor i krytyk są realizowane jako dwie odrębne sieci MLP (parametr `policy_kwargs.net_arch=dict(pi=..., vf=...)`); aktor wystawia rozkład normalny nad akcjami (średnia z sieci, $\log\sigma$ jako wyuczany parametr niezależny od stanu), a krytyk – skalarne oszacowanie wartości stanu. PPO jest naturalnym wyborem dla przestrzeni ciągłych: stabilny dzięki ograniczeniu kroku polityki, dobrze radzi sobie z gęstym sygnałem nagrody (reward shaping w naszym środowisku), nie wymaga drogiego *target network* jak SAC/TD3 i ma efektywną implementację w SB3.

In [ ]:
import os
import sys
import json
import glob
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from env.warehouse_robot_env import WarehouseRobotEnv, LAYOUT_POOL_5
from train.train_runs import CONFIGS

LOG_ROOT = ROOT / 'logs' / 'projekt5'
MODEL_ROOT = ROOT / 'models' / 'projekt5'
SEEDS = list(range(10))
TOTAL_STEPS = 200_000

print('Configi:', list(CONFIGS))
print('Logi z:', LOG_ROOT)
print('Modele:', MODEL_ROOT)

## 3. Hiperparametry — 3 zestawy

Trzy konfiguracje pokrywają jednocześnie wymaganie z poziomu 4pkt. (różne hiperparametry) i z 6pkt. (dwie architektury sieci):

| Config | Architektura `pi=vf` | `learning_rate` | Pozostałe |
|---|---|---|---|
| `small_lr3e4` | `[64, 64]` (mała) | $3\times10^{-4}$ | defaulty PPO (γ=0.99, GAE λ=0.95, `n_steps`=2048, `batch_size`=64, clip=0.2) |
| `big_lr3e4`   | `[256, 256]` (duża) | $3\times10^{-4}$ | jak wyżej |
| `small_lr1e4` | `[64, 64]` | $1\times10^{-4}$ | jak wyżej |

Para `small_lr3e4` ↔ `big_lr3e4` różni się **tylko architekturą** → porównanie sieci.<br/>
Para `small_lr3e4` ↔ `small_lr1e4` różni się **tylko learning rate** → porównanie LR.

Każda konfiguracja: **10 niezależnych uruchomień** z różnymi seedami (0–9), **200 000 kroków czasowych** na uruchomienie. Razem 30 treningów.

In [ ]:
# Wypisanie pełnej konfiguracji SB3 dla potwierdzenia
for name, cfg in CONFIGS.items():
    print(f'--- {name} ---')
    for k, v in cfg.items():
        print(f'  {k}: {v}')

## 4. Architektury sieci — schematy

Polityka PPO w SB3 (`MlpPolicy`) używa **dwóch odrębnych sieci MLP**:
* **Actor (policy)** $\pi_\theta$: $\mathbb{R}^{17} \to \mathbb{R}^{2}$, wystawia średnią rozkładu normalnego nad akcjami. Logarytmiczne odchylenie standardowe $\log\sigma$ jest osobnym, **niezależnym od stanu** parametrem wyuczanym razem z resztą wag.
* **Critic (value)** $V_\phi$: $\mathbb{R}^{17} \to \mathbb{R}$, oszacowanie wartości stanu.

Funkcja aktywacji w warstwach ukrytych: **`tanh`** (domyślne w SB3). Wyjścia są **liniowe**.

* **Wejście (17-D obs):** [x, y, $v_x$, $v_y$, carrying, $\Delta x_\text{target}$, $\Delta y_\text{target}$, $d_\text{target}$, $s_0..s_7$ (8 czujników), $n_\text{pkg-left}$].
* **Wyjście actora:** dwie składowe $(\mu_x, \mu_y) \in \mathbb{R}^2$, akcja samplowana jako $a \sim \mathcal{N}(\mu, \sigma^2)$, następnie klipowana do $[-1, 1]^2$.
* **Wyjście krytyka:** skalar $V(s) \in \mathbb{R}$.

In [ ]:
def draw_architecture(ax, hidden_sizes, title):
    """Rysuje schemat sieci actor + critic z hidden_sizes (dla obu głowic)."""
    layers_left = [('Wejście\nobs ∈ ℝ¹⁷', 17, '#e8f1ff')]
    for h in hidden_sizes:
        layers_left.append((f'Dense {h}\ntanh', h, '#cfe2ff'))
    actor_out = ('Dense 2 (linear)\nμ akcji', 2, '#ffe6b3')
    critic_out = ('Dense 1 (linear)\nV(s)', 1, '#d9f2d0')
    logstd = ('log σ\n(parametr 2-D,\nniezależny od stanu)', 2, '#ffd9d9')

    x_main = 0.4
    y_top = 1.0
    dy = 0.13
    box_h = 0.085
    box_w = 0.32

    centers = []
    for i, (label, _, color) in enumerate(layers_left):
        y = y_top - i * dy
        ax.add_patch(FancyBboxPatch((x_main - box_w/2, y - box_h/2), box_w, box_h,
                                    boxstyle='round,pad=0.01', linewidth=1.0,
                                    edgecolor='#3b3b3b', facecolor=color))
        ax.text(x_main, y, label, ha='center', va='center', fontsize=9)
        centers.append((x_main, y))

    for (x1, y1), (x2, y2) in zip(centers[:-1], centers[1:]):
        ax.add_patch(FancyArrowPatch((x1, y1 - box_h/2), (x2, y2 + box_h/2),
                                     arrowstyle='-|>', mutation_scale=10,
                                     color='#555'))

    last_x, last_y = centers[-1]
    y_branch = last_y - dy
    actor_x, critic_x = 0.18, 0.62
    for branch_x, (label, _, color) in [(actor_x, actor_out), (critic_x, critic_out)]:
        ax.add_patch(FancyBboxPatch((branch_x - box_w/2, y_branch - box_h/2), box_w, box_h,
                                    boxstyle='round,pad=0.01', linewidth=1.0,
                                    edgecolor='#3b3b3b', facecolor=color))
        ax.text(branch_x, y_branch, label, ha='center', va='center', fontsize=9)
        ax.add_patch(FancyArrowPatch((last_x, last_y - box_h/2), (branch_x, y_branch + box_h/2),
                                     arrowstyle='-|>', mutation_scale=10, color='#555'))

    logstd_x = 0.18
    ax.add_patch(FancyBboxPatch((logstd_x - box_w/2, y_branch - dy - box_h/2),
                                box_w, box_h,
                                boxstyle='round,pad=0.01', linewidth=1.0,
                                edgecolor='#3b3b3b', facecolor=logstd[2]))
    ax.text(logstd_x, y_branch - dy, logstd[0], ha='center', va='center', fontsize=8)

    sampler_y = y_branch - 2*dy
    ax.add_patch(FancyBboxPatch((actor_x - box_w/2, sampler_y - box_h/2), box_w, box_h,
                                boxstyle='round,pad=0.01', linewidth=1.0,
                                edgecolor='#3b3b3b', facecolor='#fff3cd'))
    ax.text(actor_x, sampler_y,
            'a ~ N(μ, σ²)\nclip → [−1, 1]²',
            ha='center', va='center', fontsize=8)
    ax.add_patch(FancyArrowPatch((actor_x, y_branch - box_h/2),
                                 (actor_x, sampler_y + box_h/2),
                                 arrowstyle='-|>', mutation_scale=10, color='#555'))
    ax.add_patch(FancyArrowPatch((logstd_x, y_branch - dy - box_h/2),
                                 (actor_x, sampler_y + box_h/2),
                                 arrowstyle='-|>', mutation_scale=10, color='#555',
                                 connectionstyle='arc3,rad=0.2'))

    ax.text(actor_x, sampler_y - dy*0.85, 'AKCJA', ha='center', va='top',
            fontsize=10, fontweight='bold', color='#7a5500')
    ax.text(critic_x, y_branch - dy*0.85, 'WARTOŚĆ STANU', ha='center', va='top',
            fontsize=10, fontweight='bold', color='#356b2a')

    ax.set_xlim(0, 0.8)
    ax.set_ylim(sampler_y - 2*dy, y_top + dy*0.5)
    ax.axis('off')
    ax.set_title(title, fontsize=12, pad=8)


fig, axes = plt.subplots(1, 2, figsize=(13, 8))
draw_architecture(axes[0], [64, 64], 'Architektura A — small ([64, 64])')
draw_architecture(axes[1], [256, 256], 'Architektura B — big ([256, 256])')
plt.tight_layout()
plt.show()

In [ ]:
# Liczba parametrów obu architektur (sanity check)
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

for name, cfg in CONFIGS.items():
    vec = DummyVecEnv([lambda: WarehouseRobotEnv(layout_pool=LAYOUT_POOL_5)])
    m = PPO('MlpPolicy', vec, **cfg, verbose=0)
    n_params = sum(p.numel() for p in m.policy.parameters() if p.requires_grad)
    print(f'{name:14s}  |  {n_params:>7d} parametrów  '
          f'(arch={cfg["policy_kwargs"]["net_arch"]})')

## 5. Krzywe uczenia — agregacja po 10 seedach

Dla każdego z 30 treningów `Monitor` zapisał `monitor.csv` z nagrodą i długością każdego epizodu. Krok czasowy końca epizodu *i* uzyskujemy jako $\sum_{j\le i} l_j$. Dla każdego configu interpolujemy nagrody wszystkich 10 seedów na wspólną siatkę kroków czasowych $\{0, 2000, 4000, ..., 200000\}$ i liczymy średnią ± odchylenie standardowe.

In [ ]:
def load_monitor(path: Path) -> pd.DataFrame:
    """Wczytuje plik monitor.csv (pierwsza linia to JSON metadata)."""
    df = pd.read_csv(path, skiprows=1)
    df['cumsteps'] = df['l'].cumsum()
    return df


def collect_curves(config_name: str, seeds=SEEDS, smooth_window: int = 20):
    """Zwraca listę DataFrame'ów per seed (każdy z kolumnami: cumsteps, r, r_smooth)."""
    curves = []
    for s in seeds:
        f = LOG_ROOT / config_name / f'seed_{s:02d}' / 'monitor.csv'
        if not f.exists():
            continue
        df = load_monitor(f)
        df['r_smooth'] = df['r'].rolling(smooth_window, min_periods=1).mean()
        curves.append(df)
    return curves


def aggregate_on_grid(curves, grid: np.ndarray) -> tuple[np.ndarray, np.ndarray, int]:
    """Interpoluje r_smooth każdego seeda na wspólną siatkę kroków, zwraca mean, std, n."""
    stacked = []
    for df in curves:
        if len(df) < 2:
            continue
        # interpolacja liniowa; poza zakresem -> NaN
        y = np.interp(grid, df['cumsteps'].values, df['r_smooth'].values,
                      left=np.nan, right=np.nan)
        stacked.append(y)
    if not stacked:
        return np.full_like(grid, np.nan, dtype=float), np.zeros_like(grid, dtype=float), 0
    arr = np.vstack(stacked)
    return np.nanmean(arr, axis=0), np.nanstd(arr, axis=0), arr.shape[0]


GRID = np.linspace(0, TOTAL_STEPS, 200)
agg = {name: aggregate_on_grid(collect_curves(name), GRID) for name in CONFIGS}
for name, (m, s, n) in agg.items():
    if n == 0:
        print(f'  {name:14s}  — brak logów (uruchom train/run_all.py)')
    else:
        print(f'  {name:14s}  | {n}/{len(SEEDS)} seedów  | '
              f'final reward ≈ {np.nanmean(m[-10:]):+.1f} ± {np.nanmean(s[-10:]):.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
colors = {'small_lr3e4': '#1f77b4', 'big_lr3e4': '#d62728', 'small_lr1e4': '#2ca02c'}
any_data = False
for name, (mean, std, n) in agg.items():
    if n == 0:
        continue
    any_data = True
    c = colors[name]
    ax.plot(GRID, mean, color=c, lw=2.0, label=f'{name} (n={n})')
    ax.fill_between(GRID, mean - std, mean + std, color=c, alpha=0.18)
ax.set_xlabel('Krok czasowy')
ax.set_ylabel('Nagroda epizodu (średnia kroczona, okno 20 ep.)')
ax.set_title('Krzywe uczenia — średnia ± odch. std. po 10 uruchomieniach na config')
ax.grid(alpha=0.25)
if any_data:
    ax.legend(loc='lower right')
else:
    ax.text(0.5, 0.5, 'Brak logów. Uruchom:\nuv run python train/run_all.py --steps 200000 --parallel 8',
            ha='center', va='center', transform=ax.transAxes,
            fontsize=12, color='#888',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#f5f5f5', edgecolor='#bbb'))
plt.tight_layout()
plt.show()

## 6. Czas na krok czasowy środowiska

Mierzymy średni czas wykonania pojedynczego `env.step(...)` (przy losowych akcjach) — istotny, bo determinuje *wall-clock* treningu.

In [ ]:
def benchmark_step(n_steps: int = 10_000, n_warmup: int = 100) -> dict:
    env = WarehouseRobotEnv(layout_pool=LAYOUT_POOL_5)
    env.reset(seed=0)
    # rozgrzewka
    for _ in range(n_warmup):
        _, _, term, trunc, _ = env.step(env.action_space.sample())
        if term or trunc:
            env.reset()
    # pomiar
    t0 = time.perf_counter()
    n_episodes_seen = 0
    for _ in range(n_steps):
        _, _, term, trunc, _ = env.step(env.action_space.sample())
        if term or trunc:
            env.reset()
            n_episodes_seen += 1
    dt = time.perf_counter() - t0
    env.close()
    return {
        'steps': n_steps,
        'wall_s': dt,
        'us_per_step': dt / n_steps * 1e6,
        'steps_per_s': n_steps / dt,
        'episodes_in_window': n_episodes_seen,
        'avg_episode_s': (dt / n_episodes_seen) if n_episodes_seen else None,
    }


bench = benchmark_step()
print(f"Średnio  {bench['us_per_step']:6.1f} µs / krok  "
      f"({bench['steps_per_s']:,.0f} kroków/s)")
if bench['avg_episode_s']:
    print(f"Średnio  {bench['avg_episode_s']*1000:6.1f} ms / epizod "
          f"(epizody w pomiarze: {bench['episodes_in_window']})")

In [ ]:
# Czas treningu na config (z meta.json) – jeśli logi już istnieją
rows = []
for name in CONFIGS:
    durs = []
    for s in SEEDS:
        meta_p = LOG_ROOT / name / f'seed_{s:02d}' / 'meta.json'
        if meta_p.exists():
            durs.append(json.loads(meta_p.read_text())['duration_s'])
    if durs:
        rows.append({
            'config': name,
            'n': len(durs),
            'mean_train_s': float(np.mean(durs)),
            'mean_train_min': float(np.mean(durs)) / 60,
            'us_per_step_total': float(np.mean(durs)) / TOTAL_STEPS * 1e6,
        })
if rows:
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print('Brak meta.json — uruchom train/run_all.py.')

## 7. Dyskusja — który zestaw hiperparametrów jest najlepszy?

**[Sekcja do uzupełnienia po treningu]** Spodziewamy się następujących obserwacji:

* **`big_lr3e4` vs `small_lr3e4`** — większa sieć może mieć lekko wyższy końcowy reward, ale wolniej zbiega ze względu na większą liczbę parametrów do nauczenia (~10× więcej wag). Przy 17-wymiarowej obserwacji większa pojemność nie jest konieczna, więc różnica może być niewielka.
* **`small_lr1e4` vs `small_lr3e4`** — mniejszy learning rate spowalnia uczenie; przy 200k kroków `small_lr1e4` prawdopodobnie nie zdąży osiągnąć plateau, ale może mieć stabilniejszą krzywą (mniejsze odchylenie między seedami).

Po uruchomieniu treningu należy uzupełnić tę sekcję rzeczywistymi liczbami z krzywych powyżej, w szczególności:
* końcowa średnia nagroda ± std dla każdego configu,
* prędkość uczenia (kiedy każdy config osiąga 0 / 200 / 400 nagrody),
* wnioski o roli architektury vs learning rate.

## 8. Zadanie na 8 punktów — deterministyczna ewaluacja najlepszego agenta

Po analizie krzywych wybieramy najlepszy config i trenujemy go jeszcze raz na **1 000 000 kroków czasowych** (skrypt `train/train_best_long.py --config <best>`), żeby uzyskać dobrze nauczonego agenta. Następnie uruchamiamy go z `deterministic=True` (polityka zwraca średnią $\mu$ rozkładu, bez próbkowania) i porównujemy uzyskane nagrody z krzywą uczenia.

In [ ]:
# Wybieramy config z najwyższą końcową średnią nagrodą — automatycznie po danych.
best_config = None
if any(n > 0 for *_, n in agg.values()):
    best_config = max(
        (n for n, (m, s, c) in agg.items() if c > 0),
        key=lambda n: float(np.nanmean(agg[n][0][-10:]))
    )
    print(f'Najlepszy config wg końcówki krzywej: {best_config}')
else:
    # fallback: gdy logów nie ma, ustaw ręcznie po treningu
    best_config = 'small_lr3e4'
    print(f'(Brak logów; domyślny wybór: {best_config}. Po treningu zmień ręcznie.)')

In [ ]:
# Wczytanie długo-trenowanego modelu i deterministyczny eval.
from stable_baselines3 import PPO

model_path = MODEL_ROOT / f'best_long_{best_config}.zip'
model = None
if model_path.exists():
    model = PPO.load(model_path)
    print(f'Wczytano {model_path.name}')
else:
    # fallback: weź best_long jeśli inny, lub finalny z najlepszego seeda 200k
    candidates = sorted(MODEL_ROOT.glob('best_long_*.zip')) if MODEL_ROOT.exists() else []
    if candidates:
        model = PPO.load(candidates[0])
        print(f'Brak {model_path.name}; wczytano {candidates[0].name}')
    else:
        # ostatnia deska ratunku: final_model z dowolnego seeda configu
        for s in SEEDS:
            p = LOG_ROOT / best_config / f'seed_{s:02d}' / 'final_model.zip'
            if p.exists():
                model = PPO.load(p)
                print(f'Wczytano fallback {p}')
                break
if model is None:
    print('Brak modelu. Uruchom: uv run python train/train_best_long.py --config '
          f'{best_config} --steps 1000000')

In [ ]:
def evaluate_deterministic(model, n_episodes=30, seed_base=10_000):
    env = WarehouseRobotEnv(layout_pool=LAYOUT_POOL_5)
    rewards, lengths, deliveries = [], [], []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=seed_base + i)
        total_r, steps, done = 0.0, 0, False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, term, trunc, info = env.step(action)
            total_r += r
            steps += 1
            done = term or trunc
        rewards.append(total_r)
        lengths.append(steps)
        deliveries.append(info['n_deliveries_total'])
    env.close()
    return {
        'rewards': np.array(rewards),
        'lengths': np.array(lengths),
        'deliveries': np.array(deliveries),
    }


eval_res = None
if model is not None:
    eval_res = evaluate_deterministic(model, n_episodes=30)
    r = eval_res['rewards']
    d = eval_res['deliveries']
    print(f"Eval det. (n={len(r)}):  reward = {r.mean():+.1f} ± {r.std():.1f}")
    print(f"                       paczki = {d.mean():.2f}/3,  sukces 3/3: "
          f"{int((d == 3).sum())}/{len(d)} epizodów")
else:
    print('Brak modelu — pomijam eval.')

In [ ]:
# Porównanie: krzywa uczenia best_configu vs deterministyczny eval
fig, ax = plt.subplots(figsize=(10, 5.5))
if best_config in agg and agg[best_config][2] > 0:
    mean, std, n = agg[best_config]
    c = colors.get(best_config, '#1f77b4')
    ax.plot(GRID, mean, color=c, lw=2.0, label=f'Trening (stochastyczny, {best_config}, n={n})')
    ax.fill_between(GRID, mean - std, mean + std, color=c, alpha=0.18)

if eval_res is not None:
    r = eval_res['rewards']
    ax.axhline(r.mean(), color='black', lw=2.0,
               label=f'Eval det. mean = {r.mean():+.1f}')
    ax.axhspan(r.mean() - r.std(), r.mean() + r.std(),
               color='black', alpha=0.10, label=f'± 1 std = {r.std():.1f}')
    # punkty z poszczególnych epizodów (po prawej krawędzi)
    xs = np.full_like(r, GRID[-1], dtype=float) + (np.random.RandomState(0).rand(len(r)) - 0.5) * (GRID[-1] * 0.02)
    ax.scatter(xs, r, color='black', s=18, alpha=0.55, zorder=3,
               label='Eval det. – pojedyncze epizody')

ax.set_xlabel('Krok czasowy')
ax.set_ylabel('Nagroda epizodu')
ax.set_title('Krzywa uczenia vs deterministyczna ewaluacja najlepszego agenta')
ax.grid(alpha=0.25)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 9. Wnioski końcowe

**[Sekcja do uzupełnienia po pełnym uruchomieniu]** Po wygenerowaniu wyników należy odpowiedzieć tutaj na pytania:

1. **Który zestaw hiperparametrów wygrał i dlaczego?** — uzasadnić w oparciu o krzywą uczenia (końcowa nagroda + rozrzut + szybkość zbieżności).
2. **Czy większa sieć dała przewagę?** — porównać `big_lr3e4` z `small_lr3e4` (sama architektura zmienia się).
3. **Czy niższy learning rate się opłaca?** — porównać `small_lr1e4` z `small_lr3e4`.
4. **Czy deterministyczna polityka osiąga wyższą nagrodę niż stochastyczna w trakcie uczenia?** — z założenia tak, bo `deterministic=True` używa $\mu$ (mode rozkładu), bez wariancji z $\sigma$; wynik podać liczbowo.
5. **Czy są przypadki kiedy deterministyczna polityka radzi sobie gorzej?** — np. utknięcie w stałej trajektorii dla wybranego layoutu z puli.

**Czas obliczeń (przybliżony):** 30 treningów × 200k kroków × ~0.6 ms/krok (zob. benchmark) ≈ 60 min jednowątkowo, ~8 min na 8 rdzeniach równolegle.